# 03 — Model 2: DiT-base + Ensemble 2 Model

> Notebook ini adalah **hasil run asli di Kaggle** (GPU T4) lengkap dengan output-nya.

**Peran dalam workflow:** model gambar kedua dengan arsitektur dan resolusi berbeda (ViT/BEiT, 224 px), supaya kesalahannya tidak sama dengan Donut. Di akhir notebook, probabilitas DiT digabung dengan Donut (bobot dicari pada holdout yang sama).

| | |
|---|---|
| Input | Data kompetisi + output notebook 02 |
| Output | `dit_best.pt`, `dit_val_probs.npy`, `dit_val_ids.csv`, `dit_test_probs.npy`, `submission.csv` |
| Durasi | ~65 menit (15 epoch x 4,3 menit) |
| Hasil (versi ini) | DiT holdout **0,8869** · Ensemble holdout **0,9107** · Public **0,89836** |

**Catatan versi:** tombol *Submit to Competition* di Kaggle menjalankan ulang notebook ini (Version 3). Karena training di GPU tidak 100% deterministik, hasilnya sedikit berbeda: DiT holdout **0,8907**, public **0,90129**. Output Version 3 inilah yang dipakai di notebook 05.

Kode modul: `src/models.py` (`DiTClassifier`, `llrd_param_groups`), `src/engine.py`, `configs/dit.yaml`.

# RVL-CDIP Document Classification — DiT-base + Donut Ensemble

**Inputs (Add Input):**
1. The competition data.
2. The Donut notebook's output (**Your Work → Notebooks → the Donut notebook**). This provides `val_predictions.csv`, `val_probs.npy` and `test_probs.npy`.

**Settings:** Accelerator **GPU T4**, Internet **ON** (downloads `microsoft/dit-base`).

**How to run:** `QUICK_RUN = True` → Run All (smoke test, a few minutes). Then `QUICK_RUN = False` → **Save Version → Save & Run All (Commit)**.

**Pipeline:** reuse Donut's exact holdout ids → cache all pages at 224px in RAM → fine-tune DiT-base with layer-wise LR decay →
evaluate on the holdout → tune the Donut/DiT ensemble weight on the holdout → `submission.csv`.

⚠️ Do **not** use `microsoft/dit-base-finetuned-rvlcdip`. It was trained on full RVL-CDIP, which likely includes these test images (leakage).

## 1. Config

In [1]:
import os, gc, math, random, time
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoModelForImageClassification, get_cosine_schedule_with_warmup

QUICK_RUN = False            # True = smoke test on a subset; False = full training run

IMG             = 224       # DiT's native fine-tuning resolution
BATCH_SIZE      = 64
EPOCHS          = 15
LR              = 2e-4      # LR of the top layers; lower layers get LR * LAYER_DECAY^depth
LAYER_DECAY     = 0.75
WEIGHT_DECAY    = 0.05
WARMUP_EPOCHS   = 1
LABEL_SMOOTHING = 0.1
DROP_PATH       = 0.1
SEED            = 42
NUM_WORKERS     = os.cpu_count() or 2

MODEL_NAME = "microsoft/dit-base"   # self-supervised pretrained only, NOT the rvlcdip-finetuned checkpoint
OUT = Path("/kaggle/working")

if QUICK_RUN:
    EPOCHS = 2

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("CPU workers:", NUM_WORKERS)

Device: cuda Tesla T4
CPU workers: 4


## 2. Locate the competition data and the Donut outputs

In [2]:
cands = sorted(Path("/kaggle/input").rglob("train.csv"))
assert cands, "train.csv not found under /kaggle/input - did you add the competition data?"
ROOT = cands[0].parent
print("Data root:", ROOT)

def find_file(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    assert len(hits) == 1, f"Expected exactly one {name} under /kaggle/input, found {len(hits)}: {hits}. Did you attach the Donut notebook output?"
    return hits[0]

DONUT_VAL_CSV    = find_file("val_predictions.csv")
DONUT_VAL_PROBS  = find_file("val_probs.npy")
DONUT_TEST_PROBS = find_file("test_probs.npy")
print("Donut outputs:", DONUT_VAL_CSV.parent)

train_df   = pd.read_csv(ROOT / "train.csv")
test_df    = pd.read_csv(ROOT / "test.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")

def resolve(rel):
    rel = str(rel)
    p1 = ROOT / rel
    if p1.exists():
        return str(p1)
    p2 = ROOT / Path(rel).parts[0] / rel      # e.g. ROOT/train/train/invoice/x.tif
    if p2.exists():
        return str(p2)
    return None

train_df["path"] = [resolve(f) for f in tqdm(train_df["filename"], desc="resolve train")]
test_df["path"]  = [resolve(f) for f in tqdm(test_df["filename"],  desc="resolve test")]
assert test_df["path"].notna().all(), "Some test files could not be located."
train_df = train_df[train_df["path"].notna()].reset_index(drop=True)

cls_map = train_df.groupby("label")["class_name"].agg(lambda s: s.unique().tolist())
assert all(len(v) == 1 for v in cls_map), "class_name <-> label is not one-to-one"
CLASSES = [cls_map[i][0] for i in sorted(cls_map.index)]
NUM_CLASSES = len(CLASSES)

# Donut probabilities, indexed by id so they can be aligned with any subset
donut_val = pd.read_csv(DONUT_VAL_CSV)
donut_val_probs  = np.load(DONUT_VAL_PROBS)
donut_test_probs = np.load(DONUT_TEST_PROBS)
assert donut_val_probs.shape == (len(donut_val), NUM_CLASSES), "val_probs.npy does not match val_predictions.csv"
assert donut_test_probs.shape == (len(test_df), NUM_CLASSES), "test_probs.npy does not match test.csv"
donut_val_p  = pd.DataFrame(donut_val_probs,  index=donut_val["id"].values)
donut_test_p = pd.DataFrame(donut_test_probs, index=test_df["id"].values)

Data root: /kaggle/input/competitions/document-image-classification-sd234504
Donut outputs: /kaggle/input/notebooks/yafifaldin/notebook6df86ec6cf


resolve train:   0%|          | 0/32003 [00:00<?, ?it/s]

resolve test:   0%|          | 0/7994 [00:00<?, ?it/s]

## 3. Split: reuse Donut's exact holdout
The holdout is taken from Donut's `val_predictions.csv` by id, so both models are evaluated on identical images.

In [3]:
holdout_ids = donut_val["id"].tolist()
va_df = train_df.set_index("id").loc[holdout_ids].reset_index()
assert (va_df["label"].values == donut_val["label"].values).all(), "Holdout labels do not match Donut's"
tr_df = train_df[~train_df["id"].isin(set(holdout_ids))].reset_index(drop=True)
te_df = test_df.copy()

if QUICK_RUN:
    tr_df = tr_df.groupby("label").sample(n=40, random_state=SEED).reset_index(drop=True)
    va_df = va_df.groupby("label").sample(n=10, random_state=SEED).reset_index(drop=True)
    te_df = te_df.head(64).copy()

print(f"train {len(tr_df)} | val {len(va_df)} | test {len(te_df)}")

train 28802 | val 3201 | test 7994


## 4. Cache all pages at 224px in RAM
Each TIFF is decoded once (about 2 GB of uint8 for all 40k pages), so training is GPU-bound instead of decode-bound. Area-average (BOX) downsampling keeps faint thin strokes as gray pixels. Unreadable train pages are dropped; unreadable val/test pages get a white placeholder.

In [4]:
def load_small(path):
    try:
        with Image.open(path) as im:
            im.seek(0)
            im = im.convert("L")
            return np.asarray(im.resize((IMG, IMG), Image.Resampling.BOX), dtype=np.uint8), True
    except Exception:
        return np.full((IMG, IMG), 255, dtype=np.uint8), False

def cache(paths, desc):
    with Pool(NUM_WORKERS) as pool:
        res = list(tqdm(pool.imap(load_small, paths, chunksize=64), total=len(paths), desc=desc))
    return np.stack([r[0] for r in res]), np.array([r[1] for r in res])

tr_arr, tr_ok = cache(tr_df["path"].tolist(), "cache train")
va_arr, va_ok = cache(va_df["path"].tolist(), "cache val")
te_arr, te_ok = cache(te_df["path"].tolist(), "cache test")

print("Unreadable - train:", (~tr_ok).sum(), "| val:", (~va_ok).sum(), "| test:", (~te_ok).sum())
tr_df, tr_arr = tr_df[tr_ok].reset_index(drop=True), tr_arr[tr_ok]
print(f"Cached: train {tr_arr.shape} | val {va_arr.shape} | test {te_arr.shape}")

cache train:   0%|          | 0/28802 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:982: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


cache val:   0%|          | 0/3201 [00:00<?, ?it/s]

cache test:   0%|          | 0/7994 [00:00<?, ?it/s]

Unreadable - train: 1 | val: 0 | test: 0
Cached: train (28801, 224, 224) | val (3201, 224, 224) | test (7994, 224, 224)


## 5. Dataset and augmentation
Light, scan-like augmentation only. There's no flip and no crop, since page layout is a class signal.

In [5]:
train_aug = T.Compose([
    T.RandomApply([T.RandomAffine(degrees=2, translate=(0.02, 0.02), scale=(0.95, 1.05), fill=1.0)], p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.3),
])

class ArrDataset(Dataset):
    def __init__(self, arr, labels, train=False):
        self.arr, self.labels, self.train = arr, labels, train
    def __len__(self):
        return len(self.arr)
    def __getitem__(self, i):
        x = torch.from_numpy(self.arr[i]).float().div_(255).unsqueeze(0)   # 1 x H x W in [0, 1]
        if self.train:
            x = train_aug(x)
        x = ((x - 0.5) / 0.5).expand(3, -1, -1).contiguous()               # DiT normalization, 3 channels
        return x, self.labels[i]

def make_loader(arr, labels, train):
    return DataLoader(ArrDataset(arr, labels, train), batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=train,
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(tr_arr, tr_df["label"].tolist(), True)
val_loader   = make_loader(va_arr, va_df["label"].tolist(), False)
test_loader  = make_loader(te_arr, [-1] * len(te_df), False)

## 6. Model: DiT-base with a new 16-class head
The "newly initialized" warning for the classifier and pooler is expected.

In [6]:
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    drop_path_rate=DROP_PATH,
    id2label=dict(enumerate(CLASSES)),
    label2id={c: i for i, c in enumerate(CLASSES)},
)
model.to(DEVICE)
print(type(model).__name__, f"| params {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  369MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

[transformers] BeitForImageClassification LOAD REPORT from: microsoft/dit-base
Key                          | Status     | 
-----------------------------+------------+-
layernorm.weight             | UNEXPECTED | 
layernorm.bias               | UNEXPECTED | 
lm_head.bias                 | UNEXPECTED | 
lm_head.weight               | UNEXPECTED | 
beit.pooler.layernorm.bias   | MISSING    | 
beit.pooler.layernorm.weight | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BeitForImageClassification | params 85.8M


## 7. Optimizer with layer-wise LR decay, scheduler, loss
Lower transformer layers keep their pretrained features (small LR), and the top layers and head adapt faster. This is the standard BEiT/DiT fine-tuning recipe.

In [7]:
def llrd_groups(model):
    n_layers = model.config.num_hidden_layers
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if ".encoder.layer." in n:
            lid = int(n.split(".encoder.layer.")[1].split(".")[0]) + 1
        elif ".embeddings." in n:
            lid = 0
        else:
            lid = n_layers + 1           # pooler, classifier, shared relative position bias
        no_wd = (p.ndim == 1 or n.endswith(".bias") or "norm" in n.lower()
                 or "position" in n or "cls_token" in n or "lambda" in n)
        key = (lid, no_wd)
        if key not in groups:
            groups[key] = {"params": [], "lr": LR * LAYER_DECAY ** (n_layers + 1 - lid),
                           "weight_decay": 0.0 if no_wd else WEIGHT_DECAY}
        groups[key]["params"].append(p)
    return list(groups.values())

optimizer = torch.optim.AdamW(llrd_groups(model))
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, len(train_loader) * WARMUP_EPOCHS, total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

## 8. Train and validate

In [8]:
@torch.no_grad()
def predict(loader):
    model.eval()
    probs = []
    for x, _ in tqdm(loader, desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(pixel_values=x.to(DEVICE, non_blocking=True)).logits
        probs.append(logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(probs)

best_acc = 0.0
for epoch in range(EPOCHS):
    model.train()
    t0, run_loss = time.time(), 0.0
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}")
    for step, (x, y) in enumerate(pbar):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            loss = criterion(model(pixel_values=x).logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()
        run_loss += loss.item()
        pbar.set_postfix(loss=f"{run_loss/(step+1):.4f}")

    val_probs = predict(val_loader)
    acc = accuracy_score(va_df["label"], val_probs.argmax(1))
    mf1 = f1_score(va_df["label"], val_probs.argmax(1), average="macro")
    print(f"epoch {epoch+1}: train_loss {run_loss/len(train_loader):.4f} | val_acc {acc:.4f} | "
          f"val_macroF1 {mf1:.4f} | {(time.time()-t0)/60:.1f} min")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), OUT / "dit_best.pt")
        print("  saved dit_best.pt")

print("Best DiT val acc:", best_acc)

epoch 1/15:   0%|          | 0/450 [00:01<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  368MB            

model.safetensors: downloading bytes:           |  0.00B            

predict:   0%|          | 0/51 [00:00<?, ?it/s]

epoch 1: train_loss 1.5631 | val_acc 0.8035 | val_macroF1 0.8063 | 4.4 min
  saved dit_best.pt


epoch 2/15:   0%|          | 0/450 [00:00<?, ?it/s]

... (37 outputs left out)

epoch 14: train_loss 0.6286 | val_acc 0.8863 | val_macroF1 0.8856 | 4.3 min


epoch 15/15:   0%|          | 0/450 [00:00<?, ?it/s]

predict:   0%|          | 0/51 [00:00<?, ?it/s]

epoch 15: train_loss 0.6261 | val_acc 0.8863 | val_macroF1 0.8856 | 4.3 min
Best DiT val acc: 0.886910340518588


## 9. DiT holdout error analysis

In [9]:
model.load_state_dict(torch.load(OUT / "dit_best.pt", map_location=DEVICE))
dit_val = predict(val_loader)
y_true = va_df["label"].values

print(classification_report(y_true, dit_val.argmax(1), labels=list(range(NUM_CLASSES)),
                            target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, dit_val.argmax(1), labels=list(range(NUM_CLASSES)))
pairs = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
print("Top DiT confusions (count, true -> predicted):")
for c, a, b in sorted(pairs, reverse=True)[:12]:
    print(f"  {c:4d}  {a} -> {b}")

np.save(OUT / "dit_val_probs.npy", dit_val)
va_df[["id", "label", "class_name"]].to_csv(OUT / "dit_val_ids.csv", index=False)   # row order of dit_val_probs.npy

predict:   0%|          | 0/51 [00:00<?, ?it/s]

                        precision    recall  f1-score   support

         advertisement      0.893     0.910     0.901       201
                budget      0.835     0.905     0.869       201
                 email      0.995     0.995     0.995       201
           file_folder      0.923     0.946     0.934       202
                  form      0.740     0.706     0.723       201
           handwritten      0.965     0.941     0.953       203
               invoice      0.857     0.879     0.868       198
                letter      0.895     0.909     0.902       197
                  memo      0.916     0.935     0.925       199
          news_article      0.944     0.858     0.899       197
          presentation      0.816     0.824     0.820       199
         questionnaire      0.847     0.821     0.833       195
                resume      0.956     0.966     0.961       203
scientific_publication      0.907     0.942     0.924       206
     scientific_report      0.753     0

## 10. Ensemble weight on the holdout
`P = w * Donut + (1 - w) * DiT`. The weight is chosen on the same holdout used for epoch selection, so the ensemble's holdout accuracy is slightly optimistic. The leaderboard is the real check.

In [10]:
d_val = donut_val_p.loc[va_df["id"]].values

rows = []
for w in np.round(np.arange(0, 1.0001, 0.05), 2):
    p = w * d_val + (1 - w) * dit_val
    rows.append((w, accuracy_score(y_true, p.argmax(1)), f1_score(y_true, p.argmax(1), average="macro")))
grid = pd.DataFrame(rows, columns=["w_donut", "val_acc", "val_macroF1"])
print(grid.to_string(index=False))

best_w = float(grid.loc[grid["val_acc"].idxmax(), "w_donut"])
print(f"\nDonut only: {grid.iloc[-1].val_acc:.4f} | DiT only: {grid.iloc[0].val_acc:.4f} | "
      f"ensemble (w_donut={best_w:.2f}): {grid['val_acc'].max():.4f}")
if best_w in (0.0, 1.0):
    print("Note: the best weight is a single model, so the ensemble adds nothing on the holdout.")

ens_val = best_w * d_val + (1 - best_w) * dit_val
cm = confusion_matrix(y_true, ens_val.argmax(1), labels=list(range(NUM_CLASSES)))
pairs = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
print("Top ensemble confusions (count, true -> predicted):")
for c, a, b in sorted(pairs, reverse=True)[:12]:
    print(f"  {c:4d}  {a} -> {b}")

 w_donut  val_acc  val_macroF1
    0.00 0.886910     0.886408
    0.05 0.887223     0.886741
    0.10 0.888472     0.887931
    0.15 0.888785     0.888275
    0.20 0.888472     0.888010
    0.25 0.889722     0.889326
    0.30 0.890659     0.890245
    0.35 0.892846     0.892350
    0.40 0.897844     0.897504
    0.45 0.900656     0.900319
    0.50 0.905342     0.905031
    0.55 0.907216     0.906971
    0.60 0.907529     0.907359
    0.65 0.910653     0.910322
    0.70 0.909716     0.909401
    0.75 0.908154     0.907931
    0.80 0.905654     0.905358
    0.85 0.903468     0.903290
    0.90 0.900344     0.900137
    0.95 0.896282     0.896058
    1.00 0.891909     0.891525

Donut only: 0.8919 | DiT only: 0.8869 | ensemble (w_donut=0.65): 0.9107
Top ensemble confusions (count, true -> predicted):
    11  scientific_report -> form
    10  form -> invoice
     9  presentation -> scientific_report
     9  invoice -> form
     8  scientific_report -> presentation
     8  scientific_report -

## 11. Predict test and write submissions
- `submission.csv`: the Donut + DiT ensemble (submit this one).
- `submission_dit_only.csv`: DiT alone, for comparison.
- `dit_test_probs.npy`: in `test.csv` order, for future ensembles.

In [11]:
dit_test = predict(test_loader)
np.save(OUT / "dit_test_probs.npy", dit_test)
d_test = donut_test_p.loc[te_df["id"]].values
ens_test = best_w * d_test + (1 - best_w) * dit_test

print(f"Donut vs DiT agreement on test: {(d_test.argmax(1) == dit_test.argmax(1)).mean():.4f}")

def write_sub(probs, name):
    pred_df = pd.DataFrame({"id": te_df["id"].values, "label": probs.argmax(1)})
    sub = sample_sub[["id"]].merge(pred_df, on="id", how="left")
    assert sub["label"].notna().all() and len(sub) == len(sample_sub), "Submission is missing ids"
    sub["label"] = sub["label"].astype(int)
    sub.to_csv(OUT / name, index=False)
    return sub

if QUICK_RUN:
    print("QUICK_RUN: skipping submission files. Pipeline OK. Set QUICK_RUN = False and commit.")
else:
    sub = write_sub(ens_test, "submission.csv")
    write_sub(dit_test, "submission_dit_only.csv")
    print(sub.head())
    print("Ensemble prediction distribution:\n", sub["label"].map(dict(enumerate(CLASSES))).value_counts().to_string())

predict:   0%|          | 0/125 [00:00<?, ?it/s]

Donut vs DiT agreement on test: 0.8857
           id  label
0  test_00001      0
1  test_00002      0
2  test_00003      1
3  test_00004      6
4  test_00005     13
Ensemble prediction distribution:
 label
file_folder               529
form                      527
presentation              521
scientific_report         510
letter                    508
handwritten               508
email                     507
budget                    506
advertisement             503
resume                    500
scientific_publication    496
specification             480
memo                      479
invoice                   477
questionnaire             472
news_article              471
